# 🧪 W4-D6 概念实验：RAG 系统实战（纯 numpy 版）

> 配套阅读：`第4周-Day6-RAG系统实战.md`（sentence-transformers + ChromaDB 版在那边）
> 原版依赖模型下载与 GPU。这个 notebook 用**字符 bigram TF-IDF 当 embedding、
> numpy 矩阵当向量库**，从零搭一条完整 RAG 链路：
> 语料 → 分块 → 向量化 → 检索 → 带引用的回答 → 检索质量评估
>
> 实验环境：纯 numpy + matplotlib，零网络依赖，原理与生产版完全同构。

## Step 1：知识库 —— 6 篇迷你技术文档

每篇一个知识点（Transformer / KV Cache / RLHF / DPO / RAG / 分块策略），
这正是后面评估问答命中时的标准答案表。

In [ ]:
import numpy as np
from collections import Counter

docs = {
    "Transformer": "Transformer 由自注意力机制构成。自注意力让每个词同时看到所有词。"
                   "多头注意力把向量切成多个子空间并行计算。位置编码为模型注入顺序信息。",
    "KV Cache":   "KV Cache 缓存推理时的 Key 和 Value 向量。使用 KV Cache 后生成速度提升数倍。"
                  "缓存会占用大量显存。GQA 分组查询注意力可减少缓存体积。",
    "RLHF":       "RLHF 是人类反馈强化学习。第一步训练奖励模型给回答打分。"
                  "第二步用 PPO 算法优化策略。奖励模型学到人类偏好。",
    "DPO":        "DPO 是直接偏好优化。DPO 不需要奖励模型。DPO 直接在偏好对上优化策略。"
                  "DPO 比 PPO 更简单稳定。",
    "RAG":        "RAG 是检索增强生成。RAG 先检索相关文档再生成回答。"
                  "RAG 能减少幻觉。重排序把最相关的文档排到前面。混合检索结合关键词与语义。",
    "分块策略":    "分块策略决定检索质量。chunk 过大信息稀释。chunk 过小上下文不足。"
                  "重叠分块避免句子被切断。按句子边界分块效果最好。",
}
doc_names = list(docs.keys())

def bigrams(s):
    return [s[i:i+2] for i in range(len(s) - 1)]

total_chars = sum(len(t) for t in docs.values())
vocab = set()
for t in docs.values():
    vocab.update(bigrams(t))
print(f"知识库：{len(docs)} 篇文档，共 {total_chars} 字，字符 bigram 词表 {len(vocab)} 维")
print("（生产版用 768 维 neural embedding，这里 400+ 维 bigram TF-IDF 承担同样角色）")

## Step 2：向量化 + 相似度检索（我们的 Embedding + 向量库）

TF-IDF 加权 → L2 归一化 → 余弦相似度 = 点积。
`D_vec` 矩阵就是向量库：检索 = 一次矩阵乘法（ChromaDB 内部也是这个，加了 ANN 索引）。

In [ ]:
def build_tfidf_vectors(texts):
    tfs = [Counter(bigrams(t)) for t in texts]
    df = Counter()
    for tf in tfs:
        df.update(set(tf))
    N = len(texts)
    vocab = sorted(df.keys())
    vidx = {t: i for i, t in enumerate(vocab)}
    idf = {t: np.log(1 + N / c) for t, c in df.items()}
    M = np.zeros((N, len(vocab)))
    for i, tf in enumerate(tfs):
        for t, f in tf.items():
            M[i, vidx[t]] = f * idf[t]
    M /= np.linalg.norm(M, axis=1, keepdims=True) + 1e-9
    return M, vidx, idf

D_vec, vidx, idf = build_tfidf_vectors(list(docs.values()))
print(f"向量库矩阵: {D_vec.shape}（文档数 × 维度），归一化后余弦 = 点积")

def search(query, top_k=3):
    tf = Counter(bigrams(query))
    q = np.zeros(D_vec.shape[1])
    for t, f in tf.items():
        if t in vidx:
            q[vidx[t]] = f * idf[t]
    q /= np.linalg.norm(q) + 1e-9
    sims = D_vec @ q
    order = np.argsort(sims)[::-1][:top_k]
    return [(doc_names[i], sims[i]) for i in order]

print("\n查询: 「KV Cache 占多少显存」")
for name, s in search("KV Cache 占多少显存"):
    print(f"  {s:.4f}  {name}")

## Step 3：分块 + 建块级索引（Retrieval 的粒度单位是块不是篇）

按句子边界聚合到 ~50 字一块（Day5 的结论），每块记住来源文档。
**块级检索**通常优于篇级检索：块短、向量更"尖锐"，命中更准。

In [ ]:
import re

def sentence_pack(text, target=50):
    sents = re.split("(?<=。)", text)
    chunks, cur = [], ""
    for s in sents:
        if cur and len(cur) + len(s) > target:
            chunks.append(cur); cur = s
        else:
            cur += s
    if cur:
        chunks.append(cur)
    return chunks

chunks, chunk_src = [], []
for name in doc_names:
    for c in sentence_pack(docs[name]):
        chunks.append(c); chunk_src.append(name)

C_vec, cvidx, cidf = build_tfidf_vectors(chunks)
print(f"分块结果: {len(chunks)} 块（原 {len(docs)} 篇），块级向量矩阵 {C_vec.shape}")
print("示例块:", chunks[0], "← 来源:", chunk_src[0])

def search_chunks(query, top_k=3):
    tf = Counter(bigrams(query))
    q = np.zeros(C_vec.shape[1])
    for t, f in tf.items():
        if t in cvidx:
            q[cvidx[t]] = f * cidf[t]
    q /= np.linalg.norm(q) + 1e-9
    sims = C_vec @ q
    order = np.argsort(sims)[::-1][:top_k]
    return [(chunks[i], chunk_src[i], sims[i]) for i in order]

print("\n查询: 「重排序有什么用」")
for c, src, s in search_chunks("重排序有什么用", 2):
    print(f"  {s:.4f}  [{src}] {c}")

## Step 4：完整 RAG 问答 —— 检索 + 带引用的抽取式回答

生产版把 top-k 块塞进 prompt 交给 LLM 生成；这里用**抽取式回答**模拟 LLM：
从最高分块里选与查询重叠最大的一句作答，并附引用 `[来源#块号]`——
引用约束正是防幻觉的关键（Day5 生成约束）。

In [ ]:
STOP = set("是什么有怎么和与在了吗用")

def best_sentence(chunk, query):
    qs = set(query) - STOP
    sents = re.split("(?<=。)", chunk)
    scored = [sum(1 for ch in set(s) if ch in qs) / max(len(qs), 1) for s in sents]
    return sents[int(np.argmax(scored))]

def rag_qa(question, top_k=3):
    print(f"❓ {question}")
    hits = search_chunks(question, top_k)
    for i, (c, src, s) in enumerate(hits):
        print(f"   📚 [{src}#{chunks.index(c)}] 相似度 {s:.3f}")
    c, src, _ = hits[0]
    print(f"   💬 答：{best_sentence(c, question)}  （引用：{src}）\n")

rag_qa("DPO 需要奖励模型吗")
rag_qa("PPO 怎么优化策略")
rag_qa("分块大小怎么选")

## Step 5：检索质量评估 —— 篇级 vs 块级，hit@1 / hit@3

8 个带标准答案的测试问题，对比篇级检索与块级检索的命中率，并分析失手原因。
预期：小语料上两者接近（主题清晰），但块级给出**可定位到段落**的引用。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

tests = [
    ("自注意力机制是什么", "Transformer"),
    ("多头注意力怎么工作", "Transformer"),
    ("KV Cache 会占显存吗", "KV Cache"),
    ("怎么减少缓存体积", "KV Cache"),
    ("奖励模型怎么训练", "RLHF"),
    ("DPO 和 PPO 哪个简单", "DPO"),
    ("RAG 怎么减少幻觉", "RAG"),
    ("句子被切断怎么办", "分块策略"),
]

def hit_rates(retriever, k):
    hits = 0
    for q, gold in tests:
        hits += int(any(src == gold for _, src in retriever(q, k)))
    return hits / len(tests)

def doc_retriever(q, k):
    return [(n, n) for n, _ in search(q, k)]

def chunk_retriever(q, k):
    return [(c, src) for c, src, _ in search_chunks(q, k)]

r1_doc = sum(int(doc_retriever(q, 1)[0][1] == g) for q, g in tests) / len(tests)
r3_doc = hit_rates(doc_retriever, 3)
r1_chk = sum(int(chunk_retriever(q, 1)[0][1] == g) for q, g in tests) / len(tests)
r3_chk = hit_rates(chunk_retriever, 3)

fig, ax = plt.subplots(figsize=(8, 4.2))
x = np.arange(2); w = 0.36
ax.bar(x - w/2, [r1_doc, r1_chk], w, label="hit@1")
ax.bar(x + w/2, [r3_doc, r3_chk], w, label="hit@3")
for i, (a, b) in enumerate([(r1_doc, r3_doc), (r1_chk, r3_chk)]):
    ax.text(i - w/2, a + 0.02, f"{a:.0%}", ha="center"); ax.text(i + w/2, b + 0.02, f"{b:.0%}", ha="center")
ax.set_xticks(x, ["篇级检索", "块级检索"]); ax.set_ylim(0, 1.15)
ax.set_ylabel("命中率"); ax.set_title("8 题基准：篇级 vs 块级检索")
ax.legend(); plt.tight_layout(); plt.show()

miss = [(q, g, chunk_retriever(q, 1)[0][1]) for q, g in tests if chunk_retriever(q, 1)[0][1] != g]
print(f"篇级: hit@1 {r1_doc:.0%} / hit@3 {r3_doc:.0%}    块级: hit@1 {r1_chk:.0%} / hit@3 {r3_chk:.0%}")
print("块级 hit@1 失手分析:", miss if miss else "全部命中 ✓")
print("\n→ 块短、向量尖锐 → 命中率高；生产系统再加 BM25 混合与重排序（Day3/Day5）")

## 结论

| 生产版组件 | 本实验等价物 | 结论 |
|---|---|---|
| sentence-transformers | 字符 bigram TF-IDF + L2 归一 | 语义弱但机制同构 |
| ChromaDB | numpy 矩阵 + 点积 | 检索 = 一次矩阵乘法 |
| 整篇入库 | 句子聚合分块（~50字） | 小语料命中率持平，引用粒度更细 |
| LLM 生成 | 抽取式回答 + 引用 | 引用约束防幻觉 |
| RAGAS 评估 | 8 题 hit@1/hit@3 | 有标注小基准先行 |

→ 深入阅读：同目录 `.md` 版本（ChromaDB + 真实 embedding 的生产版全流程）